In [ ]:
# Setup

import ctypes, json, requests
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

try:  # NixOS: make libcuda visible
    ctypes.CDLL("/run/opengl-driver/lib/libcuda.so.1", mode=ctypes.RTLD_GLOBAL)
except OSError:
    pass

DEVICE, SEED = "cuda", 1
torch.manual_seed(SEED); np.random.seed(SEED)

ENCODER = "tencent/KaLM-Embedding-Gemma3-12B-2511"
LAYER   = -19
LANGS   = ("de", "fr", "it")
OUT     = Path("rsd_head"); OUT.mkdir(exist_ok=True)
EPOCHS, BATCH, LR, WD = 5, 4096, 1e-3, 1e-2
HIDDEN, N_HEADS = 256, 5

# Training data: iSTS-2016 train split, machine-translated + LLM-projected labels (SwissGov-RSD repo).
# Built only from iSTS *train* + PAWS train -> disjoint from SwissGov-RSD dev/test and iSTS-RSD test.
ISTS_DIR   = Path("ists_projected"); ISTS_DIR.mkdir(exist_ok=True)
ISTS_URL   = "https://raw.githubusercontent.com/ZurichNLP/SwissGov-RSD/master/data/finetuning/encoder_inputs"
N_PER_LANG = 3000

DOC_CONCAT = 6


In [ ]:
# KLM encoder

tok   = AutoTokenizer.from_pretrained(ENCODER)
model = AutoModel.from_pretrained(ENCODER, dtype=torch.bfloat16, device_map=DEVICE).eval()
print(f"GPU: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

In [ ]:
# Data
# train/valid: iSTS projected (for training + all tuning). dev: SwissGov-RSD validation, EVALUATION ONLY.


def ists_split(split, lang, n=None):
    """Rows of {split}_{lang}_projected.jsonl. Each side mixes EN and target-language sentences
    (pairs are randomly swapped per sentence), which is fine: we encode both directions anyway."""
    f = ISTS_DIR / f"{split}_{lang}_projected.jsonl"
    if not f.exists():                         # requests uses certifi's CA bundle (uv's Python has none)
        r = requests.get(f"{ISTS_URL}/{f.name}", timeout=60); r.raise_for_status()
        f.write_bytes(r.content)
    rows = [json.loads(l) for l in open(f)]
    if n is not None:
        rows = [rows[i] for i in np.random.default_rng(SEED).permutation(len(rows))[:n]]
    out = []
    for r in rows:
        d = dict(lang=lang, en=r["text_a"].split(), other=r["text_b"].split(),     # "en"/"other" = side a/b
                 y_en=np.array(r["labels_a"], float), y_other=np.array(r["labels_b"], float))
        assert len(d["en"]) == len(d["y_en"]) and len(d["other"]) == len(d["y_other"])
        out.append(d)
    return out

def concat_docs(docs, m, seed=SEED):
    """Merge random groups of m docs (same language) into one longer doc; labels are concatenated alongside."""
    if m <= 1:
        return docs
    idx = np.random.default_rng(seed).permutation(len(docs))
    out = []
    for g in np.array_split(idx, max(1, len(docs) // m)):
        out.append(dict(lang=docs[g[0]]["lang"],
                        en=sum((docs[i]["en"] for i in g), []), other=sum((docs[i]["other"] for i in g), []),
                        y_en=np.concatenate([docs[i]["y_en"] for i in g]),
                        y_other=np.concatenate([docs[i]["y_other"] for i in g])))
    return out

train = [d for l in LANGS for d in concat_docs(ists_split("train", l, N_PER_LANG), DOC_CONCAT)]
valid = [d for l in LANGS for d in concat_docs(ists_split("valid", l), DOC_CONCAT)]   # long valid docs too, so
                                                                                       # tuning sees SwissGov-like length

dev = []
for r in load_dataset("ZurichNLP/SwissGov-RSD", split="validation"):
    d = dict(lang=r["subset"].split("_")[-1], group=r["page_en"],
             en=r["text_a"].split(), other=r["text_b"].split(),
             y_en=np.array(r["labels_a"], float), y_other=np.array(r["labels_b"], float))
    assert len(d["en"]) == len(d["y_en"]) and len(d["other"]) == len(d["y_other"]), r["id"]
    dev.append(d)
print(f"train {len(train)} | valid {len(valid)} | SwissGov dev (eval only) {len(dev)} document pairs")

In [ ]:
# Encode
# -> one vector per target word

@torch.no_grad()
def encode_pair(context_words, target_words):
    """The target comes second, so with causal attention every target word has seen the other text."""
    ctx  = " ".join(context_words) + "\n\n"
    ends = len(ctx) + np.cumsum([len(w) + 1 for w in target_words]) - 1   # end char of each word
    enc  = tok(ctx + " ".join(target_words), return_tensors="pt", return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")[0]
    h    = model(**enc.to(DEVICE), output_hidden_states=True).hidden_states[LAYER][0].float()
    last = offs[:, 1] - 1
    keep = (offs[:, 1] > offs[:, 0]) & (last >= len(ctx))                 # target subwords only
    word = torch.as_tensor(np.searchsorted(ends, last[keep].numpy(), side="right"), device=DEVICE)
    E = torch.zeros(len(target_words), h.shape[-1], device=DEVICE).index_add_(0, word, h[keep.to(DEVICE)])
    return F.normalize(E, dim=-1).half()          # unit length first: Gemma states overflow float16

In [ ]:
# KaLM word vectors for train

def states(name, docs):
    """Encode both directions of every pair. Returns X (fp16, CPU), y, lang, seq (text id), tokens; labels >= 0 only."""
    tag  = name if name == "dev" else f"{name}_n{N_PER_LANG}_c{DOC_CONCAT}"
    path = OUT / f"states_{tag}_{ENCODER.split('/')[-1]}_L{LAYER}.npz"
    if not path.exists():
        X, Y = [], []
        for i, d in enumerate(tqdm(docs, desc=name)):
            for ctx, tgt, lab in ((d["other"], d["en"], d["y_en"]), (d["en"], d["other"], d["y_other"])):
                X.append(encode_pair(ctx, tgt).cpu().numpy()); Y.append(lab)
            if i % 50 == 0:
                torch.cuda.empty_cache()
        np.savez(path, X=np.concatenate(X), y=np.concatenate(Y))
    D = np.load(path)
    toks, lang, seq, in_other = [], [], [], []
    for i, d in enumerate(docs):
        for s, (tgt, ctx) in enumerate(((d["en"], d["other"]), (d["other"], d["en"]))):
            c = set(ctx)
            toks += tgt; lang += [d["lang"]] * len(tgt); seq += [2 * i + s] * len(tgt); in_other += [t in c for t in tgt]
    assert len(toks) == len(D["y"]), f"{name}: token order does not match the cached rows - delete {path}"
    m = D["y"] >= 0                                               # words only (no punctuation / unlabeled)
    S = dict(X=torch.as_tensor(D["X"][m]), y=D["y"][m], lang=np.array(lang)[m], seq=np.array(seq)[m],
             toks=np.array(toks, dtype=object)[m], in_other=np.array(in_other)[m])
    assert torch.isfinite(S["X"]).all(), f"non-finite values in {path} - delete it and rerun"
    print(f"{name:>5}: {tuple(S['X'].shape)}  share equivalent {np.mean(S['y'] == 0):.1%}")
    return S

TR, VA, DV = states("train", train), states("valid", valid), states("dev", dev)

In [ ]:
# Train N_HEADS MLP heads on iSTS train

def report(name, S, pred):
    per_lang = " | ".join(f"en-{l} {100 * spearmanr(S['y'][S['lang'] == l], pred[S['lang'] == l]).statistic:.1f}"
                          for l in LANGS)
    print(f"{name:<30} all {100 * spearmanr(S['y'], pred).statistic:.1f} | {per_lang}")

def train_head(S, hidden=0, seed=SEED):
    """Standardise, train Linear (hidden=0) or a 1-hidden-layer MLP with BCE on all rows of S. Returns (head, mu, sd)."""
    torch.manual_seed(seed)
    X, y = S["X"], torch.as_tensor(S["y"], dtype=torch.float32)
    s = X[torch.randperm(len(X))[:50_000]].to(DEVICE).float()
    mu, sd = s.mean(0), s.std(0) + 1e-4
    d = X.shape[1]
    head = (torch.nn.Linear(d, 1) if hidden == 0 else
            torch.nn.Sequential(torch.nn.Linear(d, hidden), torch.nn.GELU(), torch.nn.Linear(hidden, 1))).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WD)
    for _ in range(EPOCHS):
        for b in torch.randperm(len(X)).split(BATCH):
            loss = F.binary_cross_entropy_with_logits(head((X[b].to(DEVICE).float() - mu) / sd).squeeze(-1),
                                                      y[b].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
    return head.eval(), mu, sd

@torch.no_grad()
def predict(h, Z):
    """Difference score in [0, 1] for word vectors Z (float16, on CPU)."""
    head, mu, sd = h
    return torch.cat([torch.sigmoid(head((c.to(DEVICE).float() - mu) / sd)).squeeze(-1)
                      for c in Z.split(BATCH)]).cpu().numpy()

ensemble = lambda hs, S: np.mean([predict(h, S["X"]) for h in hs], axis=0)

head_lin   = train_head(TR, hidden=0)
heads_mlp  = [train_head(TR, hidden=HIDDEN, seed=SEED + k) for k in range(N_HEADS)]
pv_lin, pv_mlp = predict(head_lin, VA["X"]), ensemble(heads_mlp, VA)
pd_lin, pd_mlp = predict(head_lin, DV["X"]), ensemble(heads_mlp, DV)

print("iSTS valid (in-domain, used for tuning)")
report("  linear", VA, pv_lin); report(f"  MLP({HIDDEN}) x{N_HEADS}", VA, pv_mlp)
print("SwissGov dev (held out, never used for training or tuning)")
report("  linear", DV, pd_lin); report(f"  MLP({HIDDEN}) x{N_HEADS}", DV, pd_mlp)

In [ ]:
# DiffAlign-style score

CENTER = TR["X"][torch.randperm(len(TR["X"]))[:50_000]].to(DEVICE).float().mean(0)   # remove the shared direction

@torch.no_grad()
def diffalign(S):
    seq, out = S["seq"], np.full(len(S["y"]), 0.5, np.float32)
    for i in np.unique(seq // 2):                          # seq is sorted: rows of side a, then side b, per doc
        a0, a1, b1 = np.searchsorted(seq, [2 * i, 2 * i + 1, 2 * i + 2])
        if a1 == a0 or b1 == a1:
            continue
        A = F.normalize(S["X"][a0:a1].to(DEVICE).float() - CENTER, dim=-1)
        B = F.normalize(S["X"][a1:b1].to(DEVICE).float() - CENTER, dim=-1)
        sim = A @ B.T
        out[a0:a1] = 1 - sim.max(1).values.cpu().numpy()
        out[a1:b1] = 1 - sim.max(0).values.cpu().numpy()
    return out

da_tr = diffalign(TR)
DA_LO, DA_HI = np.percentile(da_tr, [1, 99])            # map to ~[0, 1] with train percentiles (deployable)
da_norm = lambda d: np.clip((d - DA_LO) / (DA_HI - DA_LO), 0, 1)
dv_da, va_da = da_norm(diffalign(DV)), da_norm(diffalign(VA))

print("DiffAlign alone (unsupervised)")
report("  iSTS valid", VA, va_da); report("  SwissGov dev", DV, dv_da)

blend = lambda head, da, a: (1 - a) * head + a * da
ALPHAS = np.round(np.linspace(0, 1, 11), 1)
sc_va = {a: spearmanr(VA["y"], blend(pv_mlp, va_da, a)).statistic for a in ALPHAS}
print("\nalpha   iSTS valid   SwissGov dev")
for a in ALPHAS:
    print(f"{a:>5.1f}   {100 * sc_va[a]:>10.1f}   {100 * spearmanr(DV['y'], blend(pd_mlp, dv_da, a)).statistic:>12.1f}")

ALPHA = 0.5   # fixed equal-weight ensemble; iSTS valid always prefers the in-domain head, so it can't pick alpha
print("alpha fixed at", ALPHA)
pv_score, pd_score = blend(pv_mlp, va_da, ALPHA), blend(pd_mlp, dv_da, ALPHA)
report("blend, iSTS valid", VA, pv_score); report("blend, SwissGov dev", DV, pd_score)

In [ ]:
# Smoothing iSTS

def smooth(p, seq, k, w):
    """Blend each score with the mean of its +-k neighbours in the same text (weight w)."""
    out = p.copy()
    for s in np.unique(seq):
        i = np.where(seq == s)[0]
        nb = np.convolve(np.pad(p[i], k, mode="edge"), np.ones(2 * k + 1) / (2 * k + 1), mode="valid")
        out[i] = (1 - w) * p[i] + w * nb
    return out

grid = [(0, 0.0)] + [(k, w) for k in (1, 2, 3) for w in (0.3, 0.5, 0.7)]
score_va = {kw: spearmanr(VA["y"], smooth(pv_score, VA["seq"], *kw) if kw[0] else pv_score).statistic for kw in grid}
for kw in grid:
    p = smooth(pd_score, DV["seq"], *kw) if kw[0] else pd_score
    print(f"k={kw[0]} w={kw[1]:.1f}   iSTS valid {100 * score_va[kw]:.1f}   "
          f"SwissGov dev {100 * spearmanr(DV['y'], p).statistic:.1f}")
SMOOTH = dict(zip(("k", "w"), max(grid, key=score_va.get)))
print("chosen on iSTS valid:", SMOOTH)

# Number / name rules: check on valid only
DIGIT = __import__("re").compile(r"\d")
for nm, S in (("valid", VA), ("dev", DV)):
    dig = np.array([bool(DIGIT.search(t)) for t in S["toks"]])
    for rule, mask in (("numbers", dig & S["in_other"]),
                       ("names", np.array([t[:1].isupper() and len(t) >= 4 for t in S["toks"]]) & S["in_other"] & ~dig)):
        print(f"{nm:>5} {rule:>8}: {mask.sum():,} tokens | gold equivalent {np.mean(S['y'][mask] == 0):.1%} "
              f"(all words: {np.mean(S['y'] == 0):.1%})")

In [ ]:
# Save Labeler

LEVELS = np.array([0, .2, .4, .6, .8, 1.])

# Level shares from iSTS valid gold, per language. No SwissGov labels used.
SHARES = {l: np.array([np.mean(np.isclose(VA["y"][VA["lang"] == l], v)) for v in LEVELS]) for l in LANGS}

def to_levels(p, S):
    """Quantile cut-points computed on p itself (the corpus being labeled), using the iSTS gold shares."""
    out = np.zeros_like(p)
    for l in LANGS:
        i = S["lang"] == l
        cuts = np.quantile(p[i], np.cumsum(SHARES[l])[:-1])
        out[i] = LEVELS[np.searchsorted(cuts, p[i], side="right")]
    return out

sm = lambda p, S: smooth(p, S["seq"], **SMOOTH) if SMOOTH["k"] else p
pv_final, pd_final = sm(pv_score, VA), sm(pd_score, DV)
lv_va, lv_dv = to_levels(pv_final, VA), to_levels(pd_final, DV)

print(f"alpha {ALPHA} | smoothing {SMOOTH}")
print("iSTS valid");   report("  final continuous", VA, pv_final); report("  final 6 levels", VA, lv_va)
print("SwissGov dev"); report("  final continuous", DV, pd_final); report("  final 6 levels", DV, lv_dv)
print(f"\nshare labeled different: iSTS gold {np.mean(VA['y'] > 0):.1%} | "
      f"SwissGov gold {np.mean(DV['y'] > 0):.1%} | SwissGov predicted {np.mean(lv_dv > 0):.1%}")

score = lambda d: (5 - np.rint(5 * d)).astype(int)
conf = np.zeros((6, 6), int)
np.add.at(conf, (5 - score(DV["y"]), 5 - score(lv_dv)), 1)
print("\nSwissGov dev  gold \\ pred " + "".join(f"{s:>8}" for s in range(5, -1, -1)))
for i, s in enumerate(range(5, -1, -1)):
    print(f"{s:>25} " + "".join(f"{n:>8,}" for n in conf[i]))

torch.save({"encoder": ENCODER, "layer": LAYER, "hidden": HIDDEN, "levels": LEVELS, "shares": SHARES,
            "smooth": SMOOTH,
            "train_data": f"iSTS-2016 projected train, {N_PER_LANG}/lang, concat {DOC_CONCAT}, {LANGS}",
            "diffalign": dict(alpha=ALPHA, lo=DA_LO, hi=DA_HI, center=CENTER.cpu()),
            "heads": [(h.state_dict(), mu.cpu(), sd.cpu()) for h, mu, sd in heads_mlp]}, OUT / "labeler_ists.pt")
print("saved", OUT / "labeler_ists.pt")